# Prediction Feature Extraction and GT Comparison

This notebook performs one workflow only:
1. Extract feature tables from a run prediction JSON.
2. Compare prediction feature distributions against GT feature distributions.

All logic is imported from `src/cnt_project/features` and related src modules.

In [1]:
from pathlib import Path
import sys

# Ensure src/ is importable when running this notebook without editable install
repo_root = Path.cwd()
src_path = repo_root / "src"
if not src_path.exists():
    # Fallback if notebook kernel starts from a different working directory
    for parent in Path.cwd().resolve().parents:
        candidate = parent / "src"
        if candidate.exists() and (candidate / "cnt_project").exists():
            src_path = candidate
            break
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from cnt_project.features.comparison.comparison_pipeline import run_model_comparison_pipeline
from cnt_project.features.runners.extract_pred_cnt_features_runner import run_feature_extraction_for_run
from cnt_project.io import paths as paths_module
from cnt_project.io.paths import ProjectPaths

In [2]:
# Required input: only set the run name
PRED_RUN = "fluo_edt_edt_score_symmetry_score_elongated_score_without_smoothing_20260610_1004 - Copy"

# GT selection pinned through ProjectPaths
GT_SPLIT = "test"  # "test" or "train"
GT_JSON_OVERRIDE = None  # Optional: absolute path string or Path to a custom GT JSON

# Feature extraction options
OUTPUT_RUN = None
MAKE_DEBUG_VIZ = True
DEBUG_MAX_IMAGES = 6
DEBUG_MAX_OBJECTS = 8
DEBUG_ROW_INDEX = None  # Use None to auto-pick max line-density row

# Comparison options
MODEL_LABEL = "prediction"
COMPARISON_REPORT_NAME = "model_comparison"
METRICS_TO_COMPARE = ["line_density", "length_um", "width_um", "aspect_ratio", "orientation_angle"]
ENHANCED = True

In [3]:
P = ProjectPaths.from_here(paths_module.__file__)
P.ensure_outputs()

# Auto-resolve prediction JSON from run inference folder
pred_json_path = P.predicted_poly_json(PRED_RUN)
if not pred_json_path.exists():
    available_runs = sorted([p.name for p in P.runs_root.iterdir() if p.is_dir()]) if P.runs_root.exists() else []
    raise FileNotFoundError(
        "Prediction JSON not found: "
        f"{pred_json_path}\n"
        f"Set PRED_RUN to a valid run folder under {P.runs_root}.\n"
        f"Available runs: {available_runs[:20]}"
    )

# Resolve GT JSON via ProjectPaths, unless explicitly overridden
if GT_JSON_OVERRIDE is not None:
    GT_JSON_PATH = Path(GT_JSON_OVERRIDE)
else:
    split = GT_SPLIT.strip().lower()
    if split == "test":
        GT_JSON_PATH = P.test_coco_json
    elif split == "train":
        GT_JSON_PATH = P.train_coco_json
    else:
        raise ValueError("GT_SPLIT must be 'test' or 'train'.")

if not GT_JSON_PATH.exists():
    raise FileNotFoundError(f"GT JSON not found: {GT_JSON_PATH}")

feature_result = run_feature_extraction_for_run(
    pred_run=PRED_RUN,
    output_run=OUTPUT_RUN,
    make_debug_visualizations=MAKE_DEBUG_VIZ,
    debug_max_images=DEBUG_MAX_IMAGES,
    debug_max_objects_per_image=DEBUG_MAX_OBJECTS,
    debug_row_index=DEBUG_ROW_INDEX,
)

{
    "prediction_json": str(pred_json_path),
    "gt_json": str(GT_JSON_PATH),
    "feature_output_dir": feature_result["output_dir"],
    "n_debug_visualizations": feature_result.get("n_debug_visualizations", 0),
    "debug_visualizations": feature_result.get("debug_visualizations", []),
}

CNT feature extraction finished.
Prediction JSON: C:\Users\abd93000\PycharmProjects\cnt_project_v2\global_outputs\runs\fluo_edt_edt_score_symmetry_score_elongated_score_without_smoothing_20260610_1004 - Copy\inference\predicted_annotations_poly.json
Output directory: C:\Users\abd93000\PycharmProjects\cnt_project_v2\global_outputs\runs\fluo_edt_edt_score_symmetry_score_elongated_score_without_smoothing_20260610_1004 - Copy\eval\features\pred_cnt_features
Object-level CSV: C:\Users\abd93000\PycharmProjects\cnt_project_v2\global_outputs\runs\fluo_edt_edt_score_symmetry_score_elongated_score_without_smoothing_20260610_1004 - Copy\eval\features\pred_cnt_features\object_level_features.csv
Image-level CSV: C:\Users\abd93000\PycharmProjects\cnt_project_v2\global_outputs\runs\fluo_edt_edt_score_symmetry_score_elongated_score_without_smoothing_20260610_1004 - Copy\eval\features\pred_cnt_features\image_level_features.csv
Debug visualizations generated: 12 files
Debug directory: C:\Users\abd93000\

{'prediction_json': 'C:\\Users\\abd93000\\PycharmProjects\\cnt_project_v2\\global_outputs\\runs\\fluo_edt_edt_score_symmetry_score_elongated_score_without_smoothing_20260610_1004 - Copy\\inference\\predicted_annotations_poly.json',
 'gt_json': 'C:\\Users\\abd93000\\PycharmProjects\\cnt_project_v2\\data\\annotations_uniques\\test\\COCO_mask\\annotations.json',
 'feature_output_dir': 'C:\\Users\\abd93000\\PycharmProjects\\cnt_project_v2\\global_outputs\\runs\\fluo_edt_edt_score_symmetry_score_elongated_score_without_smoothing_20260610_1004 - Copy\\eval\\features\\pred_cnt_features',
 'n_debug_visualizations': 12,
 'debug_visualizations': ['C:\\Users\\abd93000\\PycharmProjects\\cnt_project_v2\\global_outputs\\runs\\fluo_edt_edt_score_symmetry_score_elongated_score_without_smoothing_20260610_1004 - Copy\\eval\\features\\pred_cnt_features\\debug_visualizations\\object_overlays\\400-1293-18_cd_c1k40r9_fl3_0sp9_0_image_right_overlay.png',
  'C:\\Users\\abd93000\\PycharmProjects\\cnt_project_v

In [4]:
model_dirs = {
    MODEL_LABEL: str(P.inference_dir(PRED_RUN))
}

comparison_dst_dir = str(P.report_dir(COMPARISON_REPORT_NAME))
comparison_result = run_model_comparison_pipeline(
    gt_json_path=str(GT_JSON_PATH),
    model_dirs=model_dirs,
    dst_dir=comparison_dst_dir,
    metrics_to_compare=METRICS_TO_COMPARE,
    enhanced=ENHANCED,
)

comparison_result["paths"]

{'detailed_csv': 'C:\\Users\\abd93000\\PycharmProjects\\cnt_project_v2\\global_outputs\\reports\\model_comparison\\detailed_model_comparison_results.csv',
 'stats_csv': 'C:\\Users\\abd93000\\PycharmProjects\\cnt_project_v2\\global_outputs\\reports\\model_comparison\\performance_statistics.csv',
 'pivoted_csv': 'C:\\Users\\abd93000\\PycharmProjects\\cnt_project_v2\\global_outputs\\reports\\model_comparison\\pivoted_mean_std_performance.csv',
 'model_summary_csv': 'C:\\Users\\abd93000\\PycharmProjects\\cnt_project_v2\\global_outputs\\reports\\model_comparison\\model_performance_summary.csv',
 'summary_txt': 'C:\\Users\\abd93000\\PycharmProjects\\cnt_project_v2\\global_outputs\\reports\\model_comparison\\comparison_summary.txt'}

In [5]:
import pandas as pd

# 1) Comparison summary (GT vs prediction distributions)
display(comparison_result["performance_stats"].sort_values(["metric", "model"]).reset_index(drop=True))

# 2) Extracted feature tables from the prediction run
object_df = pd.read_csv(feature_result["object_csv"])
image_df = pd.read_csv(feature_result["image_csv"])

print(f"Object-level rows: {len(object_df)}")
print(f"Image-level rows: {len(image_df)}")

display(object_df.head(20))
display(image_df.head(20))

# 3) Quick image-level aggregate snapshot from extracted prediction features
numeric_cols = [
    c for c in image_df.columns
    if c.startswith("mean_") or c.startswith("line_density_")
 ]
if numeric_cols:
    display(image_df[numeric_cols].describe().T)

,model,metric,mean_wasserstein,std_wasserstein,mean_jensen_shannon,std_jensen_shannon,percent_no_predictions
0,prediction,aspect_ratio,3.468570,2.465734,0.555820,0.193649,0.0
1,prediction,length_um,0.151120,0.099385,0.563574,0.199476,0.0
2,prediction,line_density,1.352474,1.149951,0.309208,0.190220,0.0
3,prediction,orientation_angle,6.178019,2.758969,0.504952,0.189977,0.0
4,prediction,width_um,0.008186,0.007616,0.606750,0.190706,0.0


Object-level rows: 3769
Image-level rows: 30


,image_id,image_file,annotation_id,object_index,area_pixels2,area_um2,perimeter,perimeter_um,length,length_um,width,width_um,aspect_ratio,orientation_angle,n_connected_components
0,1293-22_3_2_20230306222551,1293-22_3_2_20230306222551,1,0,19.0,0.007248,19.421265,0.379322,7.414214,0.144809,2.310660,0.045130,3.208699,23.133015,1.0
1,1293-22_3_2_20230306222551,1293-22_3_2_20230306222551,2,1,17.0,0.006485,16.184387,0.316101,5.414214,0.105746,2.138071,0.041759,2.532289,21.811126,1.0
2,1293-22_3_2_20230306222551,1293-22_3_2_20230306222551,3,2,49.0,0.018692,42.309444,0.826356,20.899495,0.408193,2.454075,0.047931,8.516243,-25.664523,1.0
3,1293-22_3_2_20230306222551,1293-22_3_2_20230306222551,4,3,226.0,0.086212,179.638523,3.508565,92.325902,1.803240,2.853756,0.055737,32.352422,24.332586,1.0
4,1293-22_3_2_20230306222551,1293-22_3_2_20230306222551,5,4,38.0,0.014496,28.346499,0.553643,9.828427,0.191961,3.165685,0.061830,3.104676,84.611154,1.0
5,1293-22_3_2_20230306222551,1293-22_3_2_20230306222551,6,5,8.0,0.003052,10.265993,0.200508,3.414214,0.066684,2.000000,0.039062,1.707107,84.430959,1.0
6,1293-22_3_2_20230306222551,1293-22_3_2_20230306222551,7,6,32.0,0.012207,34.494931,0.673729,15.656854,0.305798,2.299019,0.044903,6.810233,65.753844,1.0
7,1293-22_3_2_20230306222551,1293-22_3_2_20230306222551,8,7,59.0,0.022507,52.480152,1.025003,25.970563,0.507238,2.564837,0.050094,10.125620,60.020667,1.0
8,1293-22_3_2_20230306222551,1293-22_3_2_20230306222551,9,8,18.0,0.006866,18.865904,0.368475,7.242641,0.141458,2.522408,0.049266,2.871320,-26.912477,1.0
9,1293-22_3_2_20230306222551,1293-22_3_2_20230306222551,10,9,10.0,0.003815,12.162112,0.237541,4.414214,0.086215,2.000000,0.039062,2.207107,-1.430675,1.0


,image_id,image_file,n_objects,cnt_density_per_um2,line_density_mean,line_density_std,line_density_max,nematic_order_parameter,nematic_director_angle_deg,von_mises_mean_orientation_deg,...,mean_area_um2,mean_perimeter,mean_perimeter_um,mean_length,mean_length_um,mean_width,mean_width_um,mean_aspect_ratio,mean_orientation_angle,mean_n_connected_components
0,1293-22_3_2_20230306222551,1293-22_3_2_20230306222551,211,8.44,8.523438,2.255081,14.0,0.366022,0.807804,0.807804,...,0.020936,48.536640,0.947981,23.112650,0.451419,2.426065,0.047384,9.154822,0.600252,1.000000
1,400-1093-w11-c1p1-befo3-10sp-15flow_left,400-1093-w11-c1p1-befo3-10sp-15flow_left,161,6.44,7.750000,2.443103,14.0,0.580352,8.878375,8.878375,...,0.041500,70.050981,1.368183,33.669966,0.657617,3.249263,0.063462,9.915866,8.298507,1.006211
2,400-1093-w11-c1p2-befo3-20sp-15flow_left,400-1093-w11-c1p2-befo3-20sp-15flow_left,263,10.52,11.156250,3.427685,20.0,0.531198,6.052434,6.052434,...,0.031236,59.806933,1.168104,28.524158,0.557112,2.859653,0.055853,9.521627,3.522205,1.000000
3,400-1093-w13-c1-p1-10sp-15flow_left,400-1093-w13-c1-p1-10sp-15flow_left,120,4.80,7.554688,2.184135,12.0,0.505518,27.784548,27.784548,...,0.034625,59.687049,1.165763,28.344277,0.553599,3.233045,0.063145,8.448123,19.030942,1.000000
4,400-1093-w16-c-1-p1-10sp-15flow_left,400-1093-w16-c-1-p1-10sp-15flow_left,270,10.80,16.300781,3.121148,23.0,0.155443,-8.195947,-8.195947,...,0.030622,56.621301,1.105885,26.551942,0.518593,3.014601,0.058879,8.426954,-0.365585,1.000000
5,400-1093-w17-c1p3-20sp-03flow_right,400-1093-w17-c1p3-20sp-03flow_right,410,16.40,25.121094,4.595874,35.0,0.194230,37.298225,37.298225,...,0.023141,50.730859,0.990837,23.625579,0.461437,2.570929,0.050213,8.786302,10.867120,1.002439
6,400-1293-17-c10k1r5_pd_sp0_9_fl0_1_right,400-1293-17-c10k1r5_pd_sp0_9_fl0_1_right,4,0.16,0.363281,0.576949,2.0,0.535588,-24.150514,-24.150514,...,0.058365,104.639069,2.043732,51.052038,0.997110,2.838992,0.055449,15.928653,-13.089780,1.000000
7,400-1293-17-c3k1r5_pd_sp0_3_fl0_1_left,400-1293-17-c3k1r5_pd_sp0_3_fl0_1_left,20,0.80,1.824219,1.393469,5.0,0.165045,-49.271220,-49.271220,...,0.047016,70.126716,1.369662,33.169091,0.647834,3.517275,0.068697,8.931538,1.717517,1.000000
8,400-1293-17-c3k1r5_pd_sp0_3_fl0_1_right,400-1293-17-c3k1r5_pd_sp0_3_fl0_1_right,12,0.48,0.519531,0.918351,4.0,0.309027,69.543138,69.543138,...,0.021489,34.810951,0.679901,15.002032,0.293008,3.644718,0.071186,4.082263,3.437213,1.000000
9,400-1293-17-c79k1r5_pd_sp0_6_fl0_1_right,400-1293-17-c79k1r5_pd_sp0_6_fl0_1_right,10,0.40,0.496094,0.795486,3.0,0.293740,42.405815,42.405815,...,0.024605,40.468862,0.790407,18.378175,0.358949,3.362177,0.065668,5.254897,29.468667,1.000000


,count,mean,std,min,25%,50%,75%,max
line_density_mean,30.0,7.821224,7.014452,0.363281,3.945312,5.910156,8.022461,29.042969
line_density_std,30.0,2.295154,1.038313,0.576949,1.752759,2.083024,2.780739,5.239024
line_density_max,30.0,13.833333,9.203011,2.000000,9.000000,11.500000,14.000000,41.000000
mean_area_pixels2,30.0,71.663164,43.985968,24.990196,38.836458,58.498374,88.545532,192.340909
mean_area_um2,30.0,0.027337,0.016779,0.009533,0.014815,0.022315,0.033777,0.073372
mean_perimeter,30.0,53.553988,16.623089,34.810951,40.996876,51.134081,59.257160,104.639069
mean_perimeter_um,30.0,1.045976,0.324670,0.679901,0.800720,0.998713,1.157366,2.043732
mean_length,30.0,25.045737,8.512852,15.002032,18.487274,23.592759,28.175012,51.052038
mean_length_um,30.0,0.489175,0.166267,0.293008,0.361080,0.460796,0.550293,0.997110
mean_width,30.0,2.794458,0.832954,2.018267,2.118789,2.487119,3.185561,4.981415
